In [ ]:
# In this lesson, you'll learn how to use Pydantic models to define tools for OpenAI's tool calling API. You'll see how to reuse your existing models to create robust, validated tool definitions, and how to handle tool calls in your Python code. This lesson builds on your UserInput and CustomerQuery models from previous lessons.

# By the end of this lesson, you'll be able to:

# Use Pydantic models to define tool schemas for OpenAI's tool calling API
# Register your tool with the API using a validated schema
# Handle tool calls and validate arguments with Pydantic
# Integrate LLM-driven workflows with your own Python functions and data sources

In [1]:
# import packages

from pydantic import BaseModel, Field, EmailStr, field_validator
from typing import Literal, List, Optional
from datetime import datetime, date
import json
from openai import OpenAI

# import anthropic
# import instructor 

from dotenv import load_dotenv
load_dotenv()

# needed by pydantic-ai
# import nest_asyncio
# nest_asyncio.apply()

True

In [2]:
# The first positional argument to Field() is the default value. When you pass ... (Ellipsis), you're explicitly 
# telling Pydantic:
# > "This field has no default — the caller must provide it."

class UserInput(BaseModel):
    name:str = Field(..., description="User's name")
    email: EmailStr = Field(..., description="User's email address")
    query: str = Field(..., description="User's query")
    order_id: Optional[str] = Field(None, description="Order ID if available (format: ABC-12345)")

    # validate order id format
    @field_validator("order_id")
    def validate_order_id(cls, order_id):
        import re 
        if order_id is None:
            return order_id
        pattern = r"^[A-Z]{3}-\d{5}$"
        if not re.match(pattern, order_id):
            raise ValueError("order_id must be in format ABC-12345(3 uppercase letters, dash, 5 digits)")
        return order_id

    purchase_date: Optional[date] = None

In [3]:
# Define the CustomerQuery model that inherits from UserInput
class CustomerQuery(UserInput):
    priority: str = Field(..., description="Priority level: low, medium, high")
    category: Literal['refund_request', 'information_request', 'other'] = Field(..., description="Query category")
    is_complaint: bool = Field(..., description="Whether this a complaint")
    tags: List[str] = Field(..., description="Relevant keyword tags")

In [5]:
def validate_user_input(user_json:str):
    """
    Validate user input from a JSON str and return a UserInput instance if valid
    """
    try:
        # model_validate_json - converts json str obj and validates it to be compatible with Pydantic
        user_input = UserInput.model_validate_json(user_json)
        print("user input validated...")
        return user_input
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

In [6]:
"""
client.beta.chat.completions.parse(...) is OpenAI's structured output endpoint — it automatically validates 
and parses the response into your Pydantic model.
response_format=CustomerQuery tells OpenAI to return JSON that matches your Pydantic schema. It uses the model's
JSON Schema under the hood.
.choices[0].message.parsed gives you back a fully instantiated CustomerQuery object,
"""


# Define a function to call an LLM using  OpenAI to create an instance of CustomerQuery
openai_client = OpenAI()

def create_customer_query(valid_user_json: str) -> CustomerQuery:
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": valid_user_json
            }
        ],
        response_format=CustomerQuery
    )
    
    response_content = response.choices[0].message.parsed
    print("CustomerQuery generated...")
    return response_content # validated instance of CustomerQuery data model

In [7]:
# try out your validation and query creation with sample input

# define user input json data 

user_input_json = '''
{
    "name": "Joe User",
    "email": "joe@example.com",
    "query": "When can I expect delivery of the headphones I ordered?",
    "order_id": "ABC-12345",
    "purchase_date": "2025-12-01"
}
'''

# validate user input and create a CustomerQuery
valid_data = validate_user_input(user_input_json).model_dump_json() # method converts a Pydantic model instance directly into a JSON-encoded string
customer_query = create_customer_query(valid_data) # instance of CustomerQuery model

print(type(CustomerQuery))
print(customer_query.model_dump_json(indent=2))

user input validated...


CustomerQuery generated...
<class 'pydantic._internal._model_construction.ModelMetaclass'>
{
  "name": "Joe User",
  "email": "joe@example.com",
  "query": "When can I expect delivery of the headphones I ordered?",
  "order_id": "ABC-12345",
  "purchase_date": "2025-12-01",
  "priority": "medium",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "delivery",
    "headphones",
    "order_status"
  ]
}


In [10]:
# define FAQ lookup tool input as a Pydantic model 
# query and tags are the 2 fields which will be expected in lookup_faq_answer func

class FAQLookupArgs(BaseModel):
    query: str = Field(..., description = "User's query")
    tags: List[str] = Field(..., description = "Relevant keyword tags from the customer query")

In [11]:
# Define Check Order Status tool input as a Pydantic model
# order_id and email are the two fields that will be used in check_order_status func

class CheckOrderStatusArgs(BaseModel):
    order_id: str = Field(..., description="Customer's order ID (format: ABC-12345)")
    email: str = Field(..., description="Customer's email address")

    @field_validator("order_id")
    def validate_order_id(cls, order_id):
        import re 
        pattern = r"[A-Z]{3}-\d{5}$"
        if not re.match(pattern, order_id):
            raise ValueError("order_id must be in format ABC-12345 (3 uppercase letters, dash, 5 digits)")
        return order_id

In [12]:
# Create a fake FAQ database as a list of entries with keywords, that you can look up with the 
# lookup_faq_answer and check_order_status funcs. 
faq_db = [
    {
        "question": "How can I reset my password?",
        "answer": "To reset your password, click 'Forgot Password' on the sign-in page and follow the instructions sent to your email.",
        "keywords": ["password", "reset", "account"]
    },
    {
        "question": "How long does shipping take?",
        "answer": "Standard shipping takes 3-5 business days. You can track your order in your account dashboard.",
        "keywords": ["shipping", "delivery", "order", "tracking"]
    },
    {
        "question": "How can I return an item?",
        "answer": "You can return any item within 30 days of purchase. Visit our returns page to start the process.",
        "keywords": ["return", "refund", "exchange"]
    },
    {
        "question": "How can I delete my account?",
        "answer": "To delete your account, go to your account settings tab and select 'delete account'.",
        "keywords": ["delete", "account", "remove"]
    }
]

# Create a fake order database
order_db = {
    "ABC-12345": {
        "status": "shipped", "estimated_delivery": "2025-12-05",
        "purchase_date": "2025-12-01", "email": "joe@example.com"
    },
    "XYZ-23456": {
        "status": "processing", "estimated_delivery": "2025-12-15",
        "purchase_date": "2025-12-10", "email": "sue@example.com"
    },
    "QWE-34567": {
        "status": "delivered", "estimated_delivery": "2025-12-20",
        "purchase_date": "2025-12-18", "email": "bob@example.com"
    }
}

In [13]:
# Define your FAQ lookup tool
def lookup_faq_answer(args: FAQLookupArgs) -> str:
    """Look up an FAQ answer by matching tags and words in query 
    to FAQ entry keywords.
    Func goes through the faq_db and search based on keywords in the user input. If it finds an answer, it 
    returns it.
    """
    query_words = set(word.lower() for word in args.query.split())
    tag_set = set(tag.lower() for tag in args.tags)
    best_match = None
    best_score = 0
    for faq in faq_db:
        keywords = set(k.lower() for k in faq["keywords"])
        score = len(keywords & tag_set) + len(keywords & query_words)
        if score > best_score:
            best_score = score
            best_match = faq
    if best_match and best_score > 0:
        return best_match["answer"]
    return "Sorry, I couldn't find an FAQ answer for your question."

In [25]:
# Define your check order status tool
def check_order_status(args: CheckOrderStatusArgs):
    """Simulate checking the status of a customer's order by 
    order_id and email.
    Func attempts to locate an order using order an order id
    """
    order = order_db.get(args.order_id)

    # if order_id not found
    if not order:
        return {
            "order_id": args.order_id,
            "status": "not found",
            "estimated_delivery": None,
            "note": "order_id not found"
        }

    # if order id matches (db and input) but order's email (db) does not match the email id from the user input
    if args.email.lower() != order.get("email", "").lower():
        return {
            "order_id": args.order_id,
            "status": order["status"],
            "estimated_delivery": order["estimated_delivery"],
            "note": "order_id found but email mismatch"
        }

    # if order_id and email match between the db and user input
    return {
        "order_id": args.order_id,
        "status": order["status"],
        "estimated_delivery": order["estimated_delivery"],
        "note": "order_id and email match"
    }

In [26]:
# define tools for your api call 

# lookup_faq_answer() needs FAQLookupArgs.model_json_schema() to be passed in
# BaseModel.model_json_schema returns a jsonable dict of a model’s schema

# 2 tools - 
# 1st tool is named - lookup_faq_answer. This function will be called if this tool is invoked.
# func lookup_faq_answer expects FAQLookupArgs as a param...

tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "lookup_faq_answer",
            "description": "Look up an FAQ answer by matching tags to FAQ entry keywords.",
            "parameters": FAQLookupArgs.model_json_schema()
            
        }
        
    },
    {
        "type": "function",
        "function": {
            "name": "check_order_status",
            "description": "Check the status of a customer's order.",
            "parameters": CheckOrderStatusArgs.model_json_schema()
            
        }
        
    }
]

In [27]:
# Define support ticket model

class OrderDetails(BaseModel):
    status: str
    estimated_delivery: str 
    note: str

# ultimate output of the system
class SupportTicket(CustomerQuery):
    recommended_next_action: Literal['escalate_to_agent', 'send_faq_response', 'send_order_status', 
        'no_action_needed'] = Field(..., description="LLM's recommended next action for support")
    
    # construct a field inside a Pydantic model, using another Pydantic model
    order_details: Optional[OrderDetails] = Field(None, description="Order details if action is send_order_status")
    
    faq_response : Optional[str] = Field(None, description="FAQ response if action is send_faq_response")
    creation_date: datetime = Field(..., description="Date and time the ticket was created")
        

In [35]:
# decide on the next support actiona using OpenAI tool calling

"""
Idea - 
- User input gets converted via the first LLM call into customer query data model 
- This is passed on to another LLM call to decide what to do, to call a tool or to construct a support 
  ticket of a different form
- By mentioning {support_ticket_schema} in system prompt, you are not saying that the returned response should 
  be in the format of SupportTicket model, because what you are doing in this LLM call is whether or not to call
  tools. But, this helps the LLM decide whether or not a tool call is a good idea in this case.

--------------
"system" Role
Purpose: Sets the AI's behavior, context, and instructions
Scope: Defines how the model should respond, what persona to adopt, constraints to follow, and overall guidelines
Visibility: The model treats this as foundational context that shapes all responses
Example use: Instructions like "You are a helpful assistant that analyzes customer data. Always respond in JSON format."

"user" Role
Purpose: Represents the actual user's input or query
Scope: The specific question, data, or task the user wants the model to process
Visibility: The model treats this as the primary input to respond to
Example use: The actual customer query data you want analyzed

------------
tool_choice="auto" means the LLM decides whether to call a tool or respond directly with text.
With tool_choice="auto", the model will:

Analyze the user's message and the available tools
Decide autonomously whether it needs to call a tool or can answer directly
Choose which tool to call if multiple are available, or call none at all
"""
def decide_next_action_with_tools(customer_query: CustomerQuery):
    support_ticket_schema = json.dumps(SupportTicket.model_json_schema(), indent=2) 
    system_prompt = f"""
        You are a helpful customer support agent. Your job is to 
        determine what support action should be taken for the customer, 
        based on the customer query and the expected fields in the 
        SupportTicket schema below. If more information on a particular 
        order_id or FAQ response would be helpful in responding to the 
        user query and can be obtained by calling a tool, call the 
        appropriate tool to get that information. If an order_id is 
        present in the query, always look up the order status to get 
        more information on the order.

        Here is the JSON schema for the SupportTicket model you must 
        use as context for what information is expected:
        {support_ticket_schema}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": str(customer_query.model_dump())}
    ]

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tool_definitions, # this lets the LLM know which tools are available to call
        tool_choice="auto" # llm will decide if it needs to call a tool or can answer without it
    )

    message = response.choices[0].message
    #print(f"type of message is {type(message)}")
    
    tool_calls = getattr(message, "tool_calls", None)

    return message, tool_calls, messages

In [40]:
# Inspect the LLM's outputs and tool calls

message, tool_calls, messages = decide_next_action_with_tools(customer_query)


# Investigate the LLM's outputs before proceeding
print(f"LLM message:\n {json.dumps(message.model_dump(), indent=2)}")
print(f"Tool calls: \n {json.dumps([call.model_dump() for call in tool_calls], indent=2)}")

# LLM made a tool call

LLM message:
 {
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_uYj1DXaGqQbHQ3yT0OuJo01z",
      "function": {
        "arguments": "{\"order_id\":\"ABC-12345\",\"email\":\"joe@example.com\"}",
        "name": "check_order_status"
      },
      "type": "function"
    }
  ]
}
Tool calls: 
 [
  {
    "id": "call_uYj1DXaGqQbHQ3yT0OuJo01z",
    "function": {
      "arguments": "{\"order_id\":\"ABC-12345\",\"email\":\"joe@example.com\"}",
      "name": "check_order_status"
    },
    "type": "function"
  }
]


In [37]:
# set up a function to call tools if the LLM wants to call tools 
# tool calls will come from LLM response

def get_tool_outputs(tool_calls):
    tool_outputs = []
    if tool_calls:
        for tool_call in tool_calls:
            if tool_call.function.name == "lookup_faq_answer":
                print("Agent requested a call to the Lookup FAQ tool.")

                # model_validate_json - converts json str obj and validates it to be compatible with Pydantic
                # pass tool call func arguments into FAQLookupArgs, use model_validate_json() to validate whether
                # the parameters passed by the LLM are valid for that tool call.
                args = FAQLookupArgs.model_validate_json(tool_call.function.arguments)
                print(f"args are {args}")

                result = lookup_faq_answer(args)
                tool_outputs.append({"tool_call_id": tool_call.id, "output": result})
                print(f"Lookup FAQ tool returned {result}")
                
            elif tool_call.function.name == "check_order_status":
                print("Agent requested a call to Check Order Status tool...")

                args = CheckOrderStatusArgs.model_validate_json(tool_call.function.arguments)
                print(f"args are {args}")
                
                result = check_order_status(args)
                tool_outputs.append({"tool_call_id": tool_call.id, "output": result})
                print(f"Check Order Status tool returned {result}")
                
    return tool_outputs

# stage 2 : Gather any needed tool outputs and generate a support ticket ()
tool_outputs = get_tool_outputs(tool_calls) # tool_outputs is a list

print(f"Tool outputs:\n {json.dumps(tool_outputs, indent=2)}") # json.dumps converts Python object into a json formatted string

Agent requested a call to Check Order Status tool...
args are order_id='ABC-12345' email='joe@example.com'
Check Order Status tool returned {'order_id': 'ABC-12345', 'status': 'shipped', 'estimated_delivery': '2025-12-05', 'note': 'order_id and email match'}
Tool outputs:
 [
  {
    "tool_call_id": "call_4nMkLDeDxQfQLkVcB1wQKcvU",
    "output": {
      "order_id": "ABC-12345",
      "status": "shipped",
      "estimated_delivery": "2025-12-05",
      "note": "order_id and email match"
    }
  }
]


In [41]:
# message is response of the llm from decide_next_action_with_tools()
# tool_outputs is a list of information about the tools called by llm inside decide_next_action_with_tools()

def generate_structured_support_ticket(customer_query: CustomerQuery, message, tool_outputs: list):

    tool_results_str = ""
    if tool_outputs:
        for out in tool_outputs:
            tool_results_str = "\n".join([f"Tool: {out['tool_call_id']} Output: {json.dumps(out['output'])}"])
    else:
        print("No tool calls were made.")

    # print("tools result str is :")
    # print(tool_results_str)

    prompt = f"""
        You are a support agent. Use all information below to 
        generate a support ticket as a validated Pydantic model.
        Customer query: {customer_query.model_dump_json(indent=2)}
        LLM message: {str(message.content)}
        Tool results: {tool_results_str}
    """

    # create the message with structured output
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages = [{"role": "user", "content":prompt}],
        response_format=SupportTicket 
    )

    support_ticket = response.choices[0].message.parsed
    support_ticket.creation_date = datetime.now()

    return support_ticket
    

In [27]:
print(str(message.content))

None


In [44]:
# create a support ticket 

print(f"customer_query is {customer_query}")
print("xxxxxxx")
print(f"message from previous llm call is {message}")
print("xxxxxxx")
print(f"tool_outputs is {tool_outputs}")
print("xxxxxxx")

support_ticket = generate_structured_support_ticket(customer_query, message, tool_outputs)
print(support_ticket.model_dump_json(indent=2))

customer_query is name='Joe User' email='joe@example.com' query='When can I expect delivery of the headphones I ordered?' order_id='ABC-12345' purchase_date=datetime.date(2025, 12, 1) priority='medium' category='information_request' is_complaint=False tags=['delivery', 'headphones', 'order_status']
xxxxxxx
message from previous llm call is ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uYj1DXaGqQbHQ3yT0OuJo01z', function=Function(arguments='{"order_id":"ABC-12345","email":"joe@example.com"}', name='check_order_status'), type='function')])
xxxxxxx
tool_outputs is [{'tool_call_id': 'call_4nMkLDeDxQfQLkVcB1wQKcvU', 'output': {'order_id': 'ABC-12345', 'status': 'shipped', 'estimated_delivery': '2025-12-05', 'note': 'order_id and email match'}}]
xxxxxxx


{
  "name": "Joe User",
  "email": "joe@example.com",
  "query": "When can I expect delivery of the headphones I ordered?",
  "order_id": "ABC-12345",
  "purchase_date": "2025-12-01",
  "priority": "medium",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "delivery",
    "headphones",
    "order_status"
  ],
  "recommended_next_action": "send_order_status",
  "order_details": {
    "status": "shipped",
    "estimated_delivery": "2025-12-05",
    "note": "order_id and email match"
  },
  "faq_response": null,
  "creation_date": "2026-05-15T15:47:35.275376"
}


In [47]:
# Full workflow : validate, query, decide, tool, and generate ticket
# Create a new pipeline 
# UserInput data model 

user_json = '''
{
    "name": "Joe User",
    "email": "joe@example.com",
    "query": "I'm really not happy with this product I bought",
    "order_id": "QWE-34567",
    "purchase_date": null
}

'''

In [48]:
# validate user input 
# user_json is str
# validate_user_input() returns UserInput instance 
# model_dump_json() method converts a Pydantic model instance directly into a JSON-encoded string
# valid_user_json is str
valid_user_json = validate_user_input(user_json).model_dump_json()

# create customer query 
# create_customer_query returns CustomerQuery instance. customer_query is instance of CustomerQuery model 
# valid_user_json is str
customer_query = create_customer_query(valid_user_json)

message, tool_calls, messages = decide_next_action_with_tools(customer_query)

tool_outputs = get_tool_outputs(tool_calls)

support_ticket = generate_structured_support_ticket(customer_query, message, tool_outputs)
print(support_ticket.model_dump_json(indent=2))

# The response has different data models 
# User Input 
 # "name": "Joe User",
 #  "email": "joe@example.com",
 #  "query": "I'm really not happy with this product I bought",
 #  "order_id": "QWE-34567",
 #  "purchase_date": null,  

# CustomerQuery
# "priority": "high",
# "category": "other",
# "is_complaint": true,
# "tags": [
# "product dissatisfaction",
# "customer complaint",
# "order issues"
# ],

# Support Ticket data model 
# "recommended_next_action": "escalate_to_agent",
# OrderDetails
# "order_details": {
# "status": "delivered",
# "estimated_delivery": "2025-12-20",
# "note": "order_id found but email mismatch"
# },
# "faq_response": null,
# "creation_date": "2023-12-01T10:30:00Z"

user input validated...


CustomerQuery generated...


Agent requested a call to Check Order Status tool...
args are order_id='QWE-34567' email='joe@example.com'
Check Order Status tool returned {'order_id': 'QWE-34567', 'status': 'delivered', 'estimated_delivery': '2025-12-20', 'note': 'order_id found but email mismatch'}


{
  "name": "Joe User",
  "email": "joe@example.com",
  "query": "I'm really not happy with this product I bought",
  "order_id": "QWE-34567",
  "purchase_date": null,
  "priority": "high",
  "category": "refund_request",
  "is_complaint": true,
  "tags": [
    "product dissatisfaction",
    "refund",
    "complaint",
    "customer support"
  ],
  "recommended_next_action": "escalate_to_agent",
  "order_details": {
    "status": "delivered",
    "estimated_delivery": "2025-12-20",
    "note": "order_id found but email mismatch"
  },
  "faq_response": null,
  "creation_date": "2026-05-15T15:52:22.755069"
}


In [50]:
# change user input and see the difference in the results
user_json_2 = '''
{
    "name": "Joe User",
    "email": "joe@example.com",
    "query": "How do I reset my password",
    "order_id": "QWE-34567",
    "purchase_date": null
}

'''

valid_user_json = validate_user_input(user_json_2).model_dump_json()

customer_query = create_customer_query(valid_user_json)

message, tool_calls, messages = decide_next_action_with_tools(customer_query)

tool_outputs = get_tool_outputs(tool_calls)

support_ticket = generate_structured_support_ticket(customer_query, message, tool_outputs)
print(support_ticket.model_dump_json(indent=2))

user input validated...


CustomerQuery generated...


Agent requested a call to the Lookup FAQ tool.
args are query='How do I reset my password' tags=['password reset', 'account help']
Lookup FAQ tool returned To reset your password, click 'Forgot Password' on the sign-in page and follow the instructions sent to your email.
Agent requested a call to Check Order Status tool...
args are order_id='QWE-34567' email='joe@example.com'
Check Order Status tool returned {'order_id': 'QWE-34567', 'status': 'delivered', 'estimated_delivery': '2025-12-20', 'note': 'order_id found but email mismatch'}


{
  "name": "Joe User",
  "email": "joe@example.com",
  "query": "How do I reset my password",
  "order_id": "QWE-34567",
  "purchase_date": null,
  "priority": "medium",
  "category": "information_request",
  "is_complaint": false,
  "tags": [
    "password reset",
    "account help"
  ],
  "recommended_next_action": "send_faq_response",
  "order_details": {
    "status": "delivered",
    "estimated_delivery": "2025-12-20",
    "note": "order_id found but email mismatch"
  },
  "faq_response": "To reset your password, please visit the 'Account Settings' page, click on 'Reset Password', and follow the instructions.",
  "creation_date": "2026-05-15T15:55:19.089275"
}
